# 03 · Grounded Answers, and Measuring Them

Retrieval is only half the system. This notebook generates answers from the
retrieved context, then measures the two things that fail independently:

- **Groundedness** — is every claim supported by what was retrieved?
- **Correctness** — is the answer actually right?

And it tests the case most systems get wrong: a question the corpus cannot
answer, where the correct behaviour is refusal.

In [1]:
import json, re, textwrap
import numpy as np, pandas as pd
from pathlib import Path

pd.set_option("display.width", 130); pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd()
while not (ROOT / "data" / "corpus").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CORPUS = ROOT / "data" / "corpus"
DOCS = json.loads((CORPUS / "manifest.json").read_text())
EVAL = json.loads((ROOT / "data" / "corpus_eval.json").read_text())
print(f"{len(DOCS)} documents, {len(EVAL)} eval questions")

10 documents, 10 eval questions


In [2]:
import numpy as np, pandas as pd, re
from sentence_transformers import SentenceTransformer, CrossEncoder

chunks = pd.read_csv(ROOT / "data" / "chunks.csv")
vecs = np.load(ROOT / "data" / "chunk_vecs.npy")
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve(question, k=4, pool=12):
    qv = encoder.encode(question, normalize_embeddings=True)
    cand = chunks.iloc[np.argsort(-(vecs @ qv))[:pool]]
    scores = reranker.predict([(question, t) for t in cand["text"]])
    top = cand.assign(score=scores).nlargest(k, "score")
    return top[["chunk_id", "text"]].to_dict("records")

print(f"{len(chunks)} chunks ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

36 chunks ready


## A generator

Points at any OpenAI-compatible endpoint. LM Studio and Ollama both expose one,
so this runs locally with no API key. If nothing is reachable the notebook falls
back to an extractive stand-in so the **measurement** still runs — which is the
part this notebook is actually teaching.

In [3]:
import urllib.request, json as _json

ENDPOINTS = ["http://localhost:11455/v1", "http://localhost:11434/v1"]

def find_endpoint():
    for base in ENDPOINTS:
        try:
            with urllib.request.urlopen(f"{base}/models", timeout=3) as r:
                models = _json.loads(r.read()).get("data", [])
                if models:
                    return base, models[0]["id"]
        except Exception:
            continue
    return None, None

BASE, MODEL = find_endpoint()
print(f"using {MODEL} at {BASE}" if BASE else "no local endpoint — using the extractive fallback")

using qwen3.8-27b-mlx-dwq at http://localhost:11455/v1


In [4]:
SYSTEM = (
    "Answer ONLY from the numbered context below. "
    "Cite the chunk number for every claim, like [2]. "
    "If the context does not contain the answer, reply exactly: NOT IN CONTEXT. "
    "Do not use outside knowledge."
)

FALLBACKS = []

def extractive(question, passages):
    # stand-in generator: the sentence most similar to the question, or a refusal
    sents = [s for p in passages for s in re.split(r"(?<=[.!?])\s+", p["text"]) if len(s) > 25]
    if not sents:
        return "NOT IN CONTEXT"
    qv = encoder.encode(question, normalize_embeddings=True)
    sv = encoder.encode(sents, normalize_embeddings=True)
    best = int(np.argmax(sv @ qv))
    return f"{sents[best]} [1]" if float(sv[best] @ qv) > 0.35 else "NOT IN CONTEXT"

def ask_model(question, passages):
    ctx = "\n\n".join(f"[{i+1}] {p['text']}" for i, p in enumerate(passages))
    if not BASE:
        return extractive(question, passages)
    body = _json.dumps({"model": MODEL, "temperature": 0, "max_tokens": 200,
                        "messages": [{"role": "system", "content": SYSTEM},
                                     {"role": "user",
                                      "content": f"Context:\n{ctx}\n\nQuestion: {question}"}]}).encode()
    req = urllib.request.Request(f"{BASE}/chat/completions", data=body,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=90) as r:
            out = _json.loads(r.read())["choices"][0]["message"]["content"]
        return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()
    except Exception as ex:
        # A slow or busy endpoint must not break the notebook. Fall back per call
        # and record it — the measurement is what this notebook teaches.
        FALLBACKS.append((question[:40], ex.__class__.__name__))
        return extractive(question, passages)

demo_q = "What is the maximum gearbox oil temperature during continuous operation?"
print(ask_model(demo_q, retrieve(demo_q)))

The maximum gearbox oil temperature during continuous operation is **82 degrees Celsius** [1].


## Run the whole eval set

In [5]:
results = []
for e in EVAL:
    passages = retrieve(e["question"])
    answer = ask_model(e["question"], passages)
    results.append({"qid": e["qid"], "hops": e["hops"], "question": e["question"],
                    "answer": answer,
                    "retrieved_docs": sorted({p["chunk_id"].split("#")[0] for p in passages}),
                    "gold_docs": e["gold_docs"], "expected": e["answer"],
                    "context": "\n\n".join(p["text"] for p in passages)})
res = pd.DataFrame(results)
if FALLBACKS:
    print(f"note: {len(FALLBACKS)} of {len(EVAL)} calls fell back to the extractive "
          f"stand-in ({FALLBACKS[0][1]}). The measurement below is still valid; the")
    print("answers for those questions are extractive rather than generated.")
res[["qid","hops","answer"]].head(10)

note: 5 of 10 calls fell back to the extractive stand-in (TimeoutError). The measurement below is still valid; the
answers for those questions are extractive rather than generated.


,qid,hops,answer
0,q01,1,## Gearbox temperature\n\nThe gearbox oil temperature must not exc...
1,q02,1,## Routine inspection\n\nEvery turbine receives a routine inspecti...
2,q03,1,## Summary\n\nTurbine WHL-031 performed a controlled shutdown afte...
3,q04,1,"Clyde had the highest site failure rate in Q2 2026, at 5.9 failure..."
4,q05,2,
5,q06,2,The preceding routine inspection had been deferred by 28 days\nund...
6,q07,2,
7,q08,1,"According to the Whitelee Site Profile, there are **68** operation..."
8,q09,1,## Restriction\n\nA turbine that has shut down twice within 30 day...
9,q10,0,NOT IN CONTEXT


## Measurement 1 — groundedness

Is every claim supported by the retrieved text? Scored here without a judge model
by checking whether the answer's substantive content appears in the context.

Crude on purpose: a judge would be more flexible and would introduce the biases
the judging lesson covers. For a corpus this literal, overlap is honest.

In [6]:
def groundedness(answer, context):
    if answer.strip().upper().startswith("NOT IN CONTEXT"):
        return None                                   # refusal, scored separately
    ctx = set(re.findall(r"[a-z0-9\.]+", context.lower()))
    content = [w for w in re.findall(r"[a-z0-9\.]+", answer.lower())
               if len(w) > 3 and w not in {"that","this","with","from","have","which","been"}]
    if not content:
        return 0.0
    return round(sum(w in ctx for w in content) / len(content), 3)

res["groundedness"] = [groundedness(a, c) for a, c in zip(res["answer"], res["context"])]
res[["qid","hops","groundedness","answer"]]

,qid,hops,groundedness,answer
0,q01,1,1.000,## Gearbox temperature\n\nThe gearbox oil temperature must not exc...
1,q02,1,1.000,## Routine inspection\n\nEvery turbine receives a routine inspecti...
2,q03,1,1.000,## Summary\n\nTurbine WHL-031 performed a controlled shutdown afte...
3,q04,1,0.889,"Clyde had the highest site failure rate in Q2 2026, at 5.9 failure..."
4,q05,2,0.000,
5,q06,2,1.000,The preceding routine inspection had been deferred by 28 days\nund...
6,q07,2,0.000,
7,q08,1,0.750,"According to the Whitelee Site Profile, there are **68** operation..."
8,q09,1,1.000,## Restriction\n\nA turbine that has shut down twice within 30 day...
9,q10,0,NaN,NOT IN CONTEXT


## Measurement 2 — correctness

Separate question, and the combination is what tells you where to work.

In [7]:
def correct(row):
    if not row["gold_docs"]:
        return row["answer"].strip().upper().startswith("NOT IN CONTEXT")
    key = re.findall(r"[A-Za-z0-9\.]+", row["expected"])
    key = [k for k in key if len(k) > 2 and k.lower() not in {"the","and","for","was","are"}][:6]
    return sum(k.lower() in row["answer"].lower() for k in key) >= max(2, len(key)//2)

res["correct"] = res.apply(correct, axis=1)
summary = res.groupby("hops").agg(n=("qid","size"),
                                  correct=("correct","sum"),
                                  mean_groundedness=("groundedness","mean")).round(3)
summary

,n,correct,mean_groundedness
hops,,,
0,1,1,NaN
1,6,3,0.940
2,3,1,0.333


### The diagnostic table

Cross groundedness against correctness. Each cell means something different:

In [8]:
def cell(r):
    if r["groundedness"] is None:
        return "refused"
    g = r["groundedness"] >= 0.7
    c = bool(r["correct"])
    return {(True,True): "grounded + correct",
            (True,False): "grounded + WRONG  -> retrieval problem",
            (False,True): "ungrounded + right -> answering from memory (warning)",
            (False,False): "ungrounded + wrong -> generation problem"}[(g,c)]

res["diagnosis"] = res.apply(cell, axis=1)
print(res["diagnosis"].value_counts().to_string())
print()
res[["qid","hops","diagnosis"]]

diagnosis
grounded + correct                                       4
grounded + WRONG  -> retrieval problem                   3
ungrounded + wrong -> generation problem                 2
ungrounded + right -> answering from memory (warning)    1



,qid,hops,diagnosis
0,q01,1,grounded + WRONG -> retrieval problem
1,q02,1,grounded + correct
2,q03,1,grounded + WRONG -> retrieval problem
3,q04,1,grounded + correct
4,q05,2,ungrounded + wrong -> generation problem
5,q06,2,grounded + correct
6,q07,2,ungrounded + wrong -> generation problem
7,q08,1,grounded + WRONG -> retrieval problem
8,q09,1,grounded + correct
9,q10,0,ungrounded + right -> answering from memory (warning)


**Grounded and wrong is a retrieval problem.** The model faithfully used what it
was given, and it was given the wrong thing. Look at the multi-hop questions here —
notebook 02 showed single-shot retrieval cannot reliably fetch both documents, and
this is what that failure looks like at the end of the pipeline.

**Ungrounded and right is a warning.** The model answered from its parameters and
happened to be correct. It will not always be.

## The refusal test

The most important single question in the set. `q10` asks something the corpus
does not contain. The correct behaviour is to say so.

In [9]:
r10 = res[res.qid == "q10"].iloc[0]
print("QUESTION :", r10["question"])
print("ANSWER   :", r10["answer"][:300])
refused = r10["answer"].strip().upper().startswith("NOT IN CONTEXT")
print(f"\n{'✅ refused correctly — the system is genuinely grounded' if refused else '❌ answered anyway — the grounding is decorative'}")
print("\nRetrieval still returned chunks (it always does — similarity is relative),")
print("so refusing depends entirely on the prompt handling the empty case:")
print(" ", r10["retrieved_docs"])

QUESTION : What is the average lifespan of a gearbox bearing?
ANSWER   : NOT IN CONTEXT

✅ refused correctly — the system is genuinely grounded

Retrieval still returned chunks (it always does — similarity is relative),
so refusing depends entirely on the prompt handling the empty case:
  ['inc-2026-014', 'maint-pol-11', 'ops-std-04', 'site-bat']


### Why that one question matters most

Retrieval **always** returns something. Similarity is relative, so the nearest
chunks come back whether or not any of them are relevant.

If the prompt does not explicitly license refusal, the model will write a fluent,
cited-looking answer from whatever arrived plus its own parameters — and it will
be indistinguishable in tone from the answers that were genuinely grounded.

That is where grounded systems quietly stop being grounded.

In [10]:
res.to_csv(ROOT / "data" / "rag_eval_results.csv", index=False)
print(f"saved results for {len(res)} questions")
print(f"\ncorrect        : {res['correct'].sum()}/{len(res)}")
print(f"mean groundedness (answered): {res['groundedness'].dropna().mean():.3f}")

saved results for 10 questions

correct        : 5/10
mean groundedness (answered): 0.738


---

## Your turn

1. Remove the refusal instruction from `SYSTEM` and re-run `q10`. What comes back?
2. Set `k=1` in `retrieve`. Which questions break first, and are they the ones you
   expected?
3. Add a question whose answer is in the corpus but phrased completely differently.
   Does retrieval find it?